# 01 · Data feasibility

**Goal:** check that the open data is good enough before building anything.
This notebook answers the Day 1 open questions in the project plan (§7).

## Part 1 — Coffee shops from OpenStreetMap

In [ ]:
import json
from pathlib import Path
import requests
import pandas as pd

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

# A rectangle around Surabaya: (south, west, north, east).
# It also catches a little of the neighbouring regencies;
# we cut everything to the official city boundary later.
BBOX = (-7.36, 112.58, -7.18, 112.86)

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

In [ ]:
def overpass(query: str, cache_name: str) -> dict:
    """Run an Overpass query once, save the result, reuse the file afterwards."""
    path = RAW / cache_name
    if path.exists():
        print(f"Loaded from cache: {path}")
        return json.loads(path.read_text())
    response = requests.post(
        OVERPASS_URL,
        data={"data": query},
        timeout=180,
        headers={"User-Agent": "surabaya-coffee-portfolio/0.1"},
    )
    response.raise_for_status()          # stops with an error if the request failed
    data = response.json()
    path.write_text(json.dumps(data))
    print(f"Downloaded {len(data['elements'])} places -> {path}")
    return data

In [ ]:
s, w, n, e = BBOX
cafe_query = f"""
[out:json][timeout:120];
(
  node["amenity"="cafe"]({s},{w},{n},{e});
  way["amenity"="cafe"]({s},{w},{n},{e});
  node["cuisine"~"coffee"]({s},{w},{n},{e});
  way["cuisine"~"coffee"]({s},{w},{n},{e});
);
out center tags;
"""
cafes_raw = overpass(cafe_query, "osm_cafes_raw.json")

In [ ]:
rows = []
for el in cafes_raw["elements"]:
    tags = el.get("tags", {})
    rows.append({
        "osm_type": el["type"],
        "osm_id":   el["id"],
        "lat":      el.get("lat") or el.get("center", {}).get("lat"),
        "lon":      el.get("lon") or el.get("center", {}).get("lon"),
        "name":     tags.get("name"),
        "brand":    tags.get("brand"),
        "amenity":  tags.get("amenity"),
        "cuisine":  tags.get("cuisine"),
    })
cafes = pd.DataFrame(rows)
cafes.head()

In [ ]:
print("Total places:      ", len(cafes))
print("Share with a name: ", round(cafes["name"].notna().mean(), 3))
print("Share with a brand:", round(cafes["brand"].notna().mean(), 3))
print("Share with cuisine:", round(cafes["cuisine"].notna().mean(), 3))
cafes["name"].value_counts().head(25)

### Findings — Part 1 (write your answers here)

1. **How many places did we get? Is it enough?** (rough guide: over 300 is comfortable, under 100 is a warning)
   - 
2. **What share have a brand tag? What does that mean for chain detection?**
   - 
3. **Which chains do you recognise in the top 25 names?**
   - 
4. **Anything that isn't a coffee shop?**
   - 